# 02 · Teacher: frozen foundation-model features + linear probes (Colab GPU)

1. Unpack the dataset (from notebook 01's zips on Drive) to local disk.
2. Extract frozen features once per backbone (cached on Drive as `.npz`).
3. Linear probes per crop: in-domain and leave-one-dataset-out, plus open-set scores and the crop gate.
4. Fusion of complementary backbones (the teacher for phase 3).

Runtime → *Change runtime type* → **T4 GPU**. Feature extraction ≈ 5–15 min per backbone; probes run on CPU in minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GOVI = '/content/drive/MyDrive/govi-ai'
DATA_ROOT = '/content/data'
FEATS = f'{GOVI}/features'

!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[train]"
!python scripts/pack_dataset.py --unpack --out $GOVI/dataset --data-root $DATA_ROOT
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Extract features
Teachers: DINOv2 (self-supervised) and CLIP (vision-language). Baselines: frozen supervised ImageNet CNNs.
Add `dinov2_l` if time allows (≈3× slower, usually the strongest single backbone).

In [ ]:
!python scripts/extract_features.py --data-root $DATA_ROOT --out $FEATS --backbones dinov2_b clip_b16 effnet_b0 convnext_b mnv4_s

## Probes: single backbones, then fusion

In [ ]:
for bb in ['effnet_b0', 'mnv4_s', 'convnext_b', 'clip_b16', 'dinov2_b']:
    !python scripts/run_probes.py --data-root $DATA_ROOT --features $FEATS --backbones {bb}
!python scripts/run_probes.py --data-root $DATA_ROOT --features $FEATS --backbones dinov2_b clip_b16

In [ ]:
import json, glob, os, pandas as pd
rows = []
for f in sorted(glob.glob(f'{FEATS}/results/probe_*.json')):
    r = json.load(open(f)); name = '+'.join(r['backbones'])
    for crop, protos in r['crops'].items():
        for p, v in protos.items():
            if 'skipped' in v: continue
            os_ = v['open_set']
            rows.append({'backbone': name, 'crop': crop, 'protocol': p,
                         'macro_f1': v['closed_set']['macro_f1'], 'ece': v['closed_set']['ece_calibrated'],
                         **{f'auroc_{k}': os_[k]['energy']['auroc'] for k in os_}})
t = pd.DataFrame(rows)
t['setting'] = t.protocol.where(t.protocol == 'split', 'lodo (mean)')
t.groupby(['crop', 'setting', 'backbone']).mean(numeric_only=True).round(3)